# Exercise 1 - Visual assessment<p>
Consider the Cork Stoppers data set. Examine the box/violin plots for each feature and examine their discriminative power 

In [1]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import plotly.express as px
import pandas as pd
from scipy import stats
import os

datasets_folder = os.path.abspath(os.path.join(os.getcwd() ,"../..")) + "/Datasets/" #Set the path to the datasets folder, which is assumed to be two levels up from the current working directory and then inside a "Datasets" folder. Adjust this path as necessary based on your actual directory structure.

Labels=["Super","Average","Poor"] 
xlsFile=pd.read_excel(datasets_folder + "CORK_STOPPERS.XLS",sheet_name="Data",
                      usecols=["C","ART","N.1","PRT","ARM","PRM","ARTG","NG","PRTG","RAAR","RAN"])
CorkStopp=pd.DataFrame(data=xlsFile.dropna())

fnames=CorkStopp.columns[1:] # Get features names

ixSuper = np.where(CorkStopp['C']==1)
ixAverage = np.where(CorkStopp['C']==2)
ixPoor = np.where(CorkStopp['C']==3)

ixs = np.array([ixSuper,ixAverage,ixPoor])

for f in fnames:
    fig = go.Figure()
    k = 0
    for ix in ixs:
        fig.add_trace(go.Violin(y = CorkStopp[f].iloc[ix[0]], name = Labels[k],
                                box_visible = True,
                                meanline_visible = True,points = 'all'))
        k = k+1
    fig.update_layout(title=f,autosize=False,width=1200,height=600,font=dict(size=18,color="black"))
    
    fig.show()

# Exercise 2 - Kruskal Wallis Test<p>

The Kruskal Wallis test assesses whether $N$ samples belonging to $K$ classes are from the same population (or from populations with continuous distribution) and have the same median for the variable being tested.<p>
Let $R_i$ denote the average of ranks for class $i$, with $n_i$ cases. Under the null hypothesis, we expect each $R_i$ will exhibit a small deviation from the average of all $R_i$,  $\bar{R}$. The test statistic is:<p>
$H =\frac{12}{N(N + 1)}\sum_{i=1}^Kn_i(R_i-\bar{R})^2$<p>
, which, under the null hypothesis has a chi-square distribution with $df = K-1$ degrees of freedom.<p>

### - Consider again the Cork Stoppers data set.<p>
#### 1. Use SciPy stats.kruskal function to evaluate the discriminative power of each feature. Rank features accourding to H.

In [2]:

X = CorkStopp.to_numpy()[:,1:]
X = X.astype(float)

Hs={} # Dictionary to store the H statistic for each feature
for i in range(np.shape(X)[1]):
    st = stats.kruskal(X[ixPoor,i].flatten(), X[ixAverage,i].flatten(), X[ixSuper,i].flatten())
    Hs[fnames[i]] = st.statistic

Hs = sorted(Hs.items(), key=lambda x: x[1],reverse=True) # Sort the features based on their H statistic in descending order

# Print the ranked features and their corresponding H statistic values
print("Ranked features") 
for f in Hs:
    print(f[0]+"-->"+str(f[1]))


Ranked features
ART-->121.58964258474009
PRM-->117.63782046335828
PRT-->115.74851860346449
ARTG-->115.21685092055695
ARM-->113.53079541575394
PRTG-->113.33179317347778
RAAR-->105.18901177731905
NG-->104.41349241044973
RAN-->94.28568910767306
N.1-->74.49926324214721


#### 2. Find the Correlation Matrix for the 5 first features of the data set. Inspect about redundancy.

In [3]:
# Compute the correlation matrix for the top 5 features based on their H statistic values
X = np.array([CorkStopp[Hs[0][0]],
            CorkStopp[Hs[1][0]],
            CorkStopp[Hs[2][0]],
            CorkStopp[Hs[3][0]],
            CorkStopp[Hs[4][0]]])
corrMat = np.corrcoef(X) 
fig = px.imshow(corrMat, text_auto=True,labels=dict(x="Features", y="Features", color="Correlation"),
                x=[Hs[0][0], Hs[1][0], Hs[2][0], Hs[3][0], Hs[4][0]],
                y=[Hs[0][0], Hs[1][0], Hs[2][0], Hs[3][0], Hs[4][0]],
                width=800, height=800,color_continuous_scale=px.colors.sequential.gray)
fig.show()

# Exercise 3 - ROC Curves<p>
#### 1-The Excel file SignalNoise.xls,  contains 100 samples of signal + noise. The arrival times of the signal impulses have a Poisson distribution. Change the value of the detection threshold and observe the changes performed on the detections. Construct a ROC curve.

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import os

datasets_folder = os.path.abspath(os.path.join(os.getcwd() ,"../..")) + "/Datasets/" #Set the path to the datasets folder, which is assumed to be two levels up from the current working directory and then inside a "Datasets" folder. Adjust this path as necessary based on your actual directory structure.

xlsFile=pd.read_excel(datasets_folder + "SIGNAL_NOISE.XLS",
                      sheet_name="Signal+noise",
                      usecols=["signal+noise","Correct detections"])
data = xlsFile.dropna()
sigNoise = data['signal+noise'].to_numpy()
y = data['Correct detections'].to_numpy()
pIx = np.where(y==1)[0] # Indices of positive events (where the correct detection is 1)


#===================Threshold Definition==================
tH=np.min(data)-0.5; # Initial Threshold at the (minimum - 0.5) to ensure that all events are classified as positive at the first threshold
mTh=np.max(data)+0.5; # Maximum threshold at the (maximum + 0.5) to ensure that all events are classified as negative at the last threshold
nTh=10; # Number of thresholds

incTh=(mTh-tH)/nTh;#Increment between thresholds

#Start figure definition
fig = go.FigureWidget()
fig.add_scatter(x=np.arange(sigNoise.shape[0]), y=sigNoise,
                    mode='lines+markers',
                    name='Negative Events')
fig.add_scatter(x=pIx, y=sigNoise[pIx],mode='markers', name='Positive Events', marker_size=15)
fig.add_scatter(x=np.arange(sigNoise.shape[0]), y=tH*np.ones(sigNoise.shape[0]),mode='lines',name='Threshold', marker_size=15)
fig.update_layout(autosize=False,width=1000,height=400)
fig.update_xaxes(title_text="Events")
fig.update_yaxes(title_text="Signal+Noise")
display(fig)
thFig = fig.data[2]

SS=[]
FPR=[]
figR = go.FigureWidget()
figR.add_scatter(x=SS, y=FPR,mode='lines+markers',name='Negative Events')

figR.update_layout(autosize=False,width=700,height=700)
figR.update_xaxes(title_text="1-SP",range=[-0.01, 1.01])
figR.update_yaxes(title_text="SS",range=[-0.01, 1.01])
ROCFig = figR.data[0]
display(figR)
# Test different thresholds
while tH<mTh: # Loop until the threshold reaches the maximum threshold
    thFig.y = tH*np.ones(sigNoise.shape[0])
    PosPredIx=np.where(sigNoise>=tH)[0]
    NegPredIx=np.where(sigNoise<tH)[0]
    
    #Compute confusiin matrix
    TP=np.shape(np.where((y[PosPredIx]==1))[0])[0] # True Positives
    TN=np.shape(np.where((y[NegPredIx]==0))[0])[0] # True Negatives
    FP=np.shape(np.where((y[PosPredIx]==0))[0])[0] # False Positives
    FN=np.shape(np.where((y[NegPredIx]==1))[0])[0] # False Negatives
    
    SSi=TP/(TP+FN) # Sensitivity
    SPi=TN/(TN+FP) # Specificity
    
    SS.append(SSi)
    FPR.append(1-SPi)
    ROCFig.x=FPR#Update figure
    ROCFig.y=SS
    
    
    tH = tH+incTh
    fig.show()

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Negative Events',
              'type': 'scatter',
              'uid': '885c56e4-07d8-4f9b-b7d5-a01adf9a32fa',
              'x': {'bdata': ('AAECAwQFBgcICQoLDA0ODxAREhMUFR' ... '9QUVJTVFVWV1hZWltcXV5fYGFiYw=='),
                    'dtype': 'i1'},
              'y': {'bdata': ('0XypVV8Ttz878kVZnR76P1DCuC/4i6' ... '3pd5PlP+pBgYNzjsc/OH98cgZPyj8='),
                    'dtype': 'f8'}},
             {'marker': {'size': 15},
              'mode': 'markers',
              'name': 'Positive Events',
              'type': 'scatter',
              'uid': 'be3ffb87-2349-4d1d-92c1-1995df2270b6',
              'x': {'bdata': 'AxUaKjQ9SlFU', 'dtype': 'i1'},
              'y': {'bdata': ('1qIabLpUEUDKxWtyKlj6P3MHsSeSex' ... '8kvNpIFUAXVRqlG/8CQL+GdQ3HFSxA'),
                    'dtype': 'f8'}},
             {'marker': {'size': 15},
              'mode': 'lines',
              'name': 'Threshold',
              'type': 's

FigureWidget({
    'data': [{'mode': 'lines+markers',
              'name': 'Negative Events',
              'type': 'scatter',
              'uid': '11387303-8fa7-4d7f-9d1f-420aaee271f4',
              'x': [],
              'y': []}],
    'layout': {'autosize': False,
               'height': 700,
               'template': '...',
               'width': 700,
               'xaxis': {'range': [-0.01, 1.01], 'title': {'text': '1-SP'}},
               'yaxis': {'range': [-0.01, 1.01], 'title': {'text': 'SS'}}}
})

#### 2-Consider the Cork Stoppers Data Set first two classes. Use the ROC curve approach to rank the features.  Rank the features and compare it with the raking by performed by Kruskal-Wallis.

In [5]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc


ixSupAvg=np.concatenate((ixSuper[0],ixAverage[0]))
y=CorkStopp['C'].to_numpy()[ixSupAvg]


roc_auc=np.zeros(fnames.shape)
i=0
for f in fnames:#Go along features
    fpr, tpr, _= roc_curve(y,CorkStopp[f].to_numpy()[ixSupAvg],pos_label=2) # Compute the false positive rate and true positive rate for the current feature using the roc_curve function from sklearn.metrics. The pos_label parameter is set to 2, which indicates that the positive class is represented by the value 2 in the target variable y.
    
    figR = go.Figure()
    figR.add_scatter(x=fpr, y=tpr,mode='lines+markers')
    figR.update_layout(autosize=False,width=700,height=700,title=dict(text=f))
    figR.update_xaxes(title_text="1-SP",range=[-0.01, 1.01])
    figR.update_yaxes(title_text="SS",range=[-0.01, 1.01])

    roc_auc[i] = auc(fpr, tpr)#Compute area under the ROC curve 
    
    figR.add_annotation(x=0.5, y=0.5,
            text="AUC: "+str(roc_auc[i]),
            showarrow=False,
            yshift=10)
    figR.show()
    i=i+1

sortIx=np.flip(np.argsort(roc_auc))#Sort using AUC
print("Sorting accourding to ROC-AUC:")
for i in sortIx:
    print(fnames[i]+"-->"+str(roc_auc[i]))
print("\n")
print("Sorting accourding to Kruskall-Wallis:")

for f in Hs:
    print(f[0]+"-->"+str(f[1]))

Sorting accourding to ROC-AUC:
ART-->0.9474
PRT-->0.9294
PRM-->0.901
ARM-->0.8872
ARTG-->0.867
PRTG-->0.8612
N.1-->0.848
NG-->0.8464
RAAR-->0.7968
RAN-->0.7881999999999999


Sorting accourding to Kruskall-Wallis:
ART-->121.58964258474009
PRM-->117.63782046335828
PRT-->115.74851860346449
ARTG-->115.21685092055695
ARM-->113.53079541575394
PRTG-->113.33179317347778
RAAR-->105.18901177731905
NG-->104.41349241044973
RAN-->94.28568910767306
N.1-->74.49926324214721
